In [19]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from ast import literal_eval # as tuple literal_eval -> convert string representation of list to actual list


In [20]:
# --- par‑plant ---------------------------------------------------------------
path_box   = Path("/home/loai/Documents/code/RSMLExtraction/Results/Measures/results_per_plant.csv")
df_plant_wid = pd.read_csv(path_box)
df_plant_wid['root_ids'] = df_plant_wid['root_ids'].apply(literal_eval)
df_plant_wid

,model,split,box,metric,time,root_ids,source,value
0,Segformer_bce_dice,Val,230629PN006,NumberOfOrgans,1,"(9, 21, 10)",Prediction,1.000000
1,Segformer_bce_dice,Val,230629PN006,NumberOfOrgans,1,"(9, 21, 10)",expertized,1.000000
2,Segformer_bce_dice,Val,230629PN006,NumberOfOrgans,1,"(9, 21, 10)",before_expertized,1.000000
3,Segformer_bce_dice,Val,230629PN006,NumberOfOrgans,1,"(13, 23, 12)",Prediction,1.000000
4,Segformer_bce_dice,Val,230629PN006,NumberOfOrgans,1,"(13, 23, 12)",expertized,1.000000
...,...,...,...,...,...,...,...,...
179563,Unet_bce_dice,Test,230629PN024,LateralRootLength,29,"(88, 65, 65)",expertized,1249.734308
179564,Unet_bce_dice,Test,230629PN024,LateralRootLength,29,"(88, 65, 65)",before_expertized,978.395032
179565,Unet_bce_dice,Test,230629PN024,LateralRootLength,29,"(22, 1, 1)",Prediction,1513.592493
179566,Unet_bce_dice,Test,230629PN024,LateralRootLength,29,"(22, 1, 1)",expertized,1722.149388


In [21]:
df_grouped = (
    df_plant_wid
    .set_index(['model','split','box','metric','root_ids','source','time'])
    .sort_index(level='time')
)
df_grouped

value
model            split box         metric           root_ids     source            time             
Segformer_bce    Test  230629PN008 Convex_Area_Hull (8, 1, 1)    Prediction        1      331.312847
                                                                 before_expertized 1      328.799340
                                                                 expertized        1      328.799340
                                                    (18, 11, 8)  Prediction        1      408.575615
                                                                 before_expertized 1      406.477047
...                                                                                              ...
Unet_dice_cldice Val   230629PN031 TotalRootLength  (65, 66, 63) before_expertized 29    2846.313710
                                                                 expertized        29    2846.313710
                                                    (83, 84, 81) Prediction        29    3381.903847
                                                                 before_expertized 29    3313.831728
                                                                 expertized        29    3296.533489

[179568 rows x 1 columns]

In [22]:
models   = df_grouped.index.get_level_values('model').unique()
splits   = df_grouped.index.get_level_values('split').unique()
boxes    = df_grouped.index.get_level_values('box').unique()
metrics  = df_grouped.index.get_level_values('metric').unique()

print("Models disponibles :", models)
print("Splits disponibles :", splits)
print("Boxes disponibles  :", boxes)
print("Metrics disponibles:", metrics)


Models disponibles : Index(['Segformer_bce', 'Segformer_bce_dice', 'Unet_bce', 'Unet_bce_dice',
       'Unet_dice', 'Unet_dice_cldice'],
      dtype='object', name='model')
Splits disponibles : Index(['Test', 'Val'], dtype='object', name='split')
Boxes disponibles  : Index(['230629PN008', '230629PN010', '230629PN018', '230629PN024',
       '230629PN027', '230629PN006', '230629PN012', '230629PN014',
       '230629PN019', '230629PN031'],
      dtype='object', name='box')
Metrics disponibles: Index(['Convex_Area_Hull', 'Intercept_curve_Area', 'LateralRootLength',
       'NumberOfLateralRoots', 'NumberOfOrgans', 'PrimaryRootLength',
       'RootDensity', 'TotalRootLength'],
      dtype='object', name='metric')


In [ ]:
def get_root_ids(model, split, box, metric):
    return (
        df_grouped
        .xs((model, split, box, metric), level=('model', 'split', 'box', 'metric'))
        .index.get_level_values('root_ids')
        .unique()
    )


def get_sources(model, split, box, metric, root):
    return (
        df_grouped
        .xs((model, split, box, metric, root), level=('model', 'split', 'box', 'metric', 'root_ids'))
        .index.get_level_values('source')
        .unique()
    )

# 4) Fonction de tracé interactive


def plot_root(model_idx, split_idx, box_idx, metric_idx, root_idx):
    model = models[model_idx]
    split = splits[split_idx]
    box = boxes[box_idx + split_idx * 5]
    metric = metrics[metric_idx]
    roots = get_root_ids(model, split, box, metric)
    root = roots[root_idx]

    df_tmp = (
        df_grouped
        .xs((model, split, box, metric, root),
            level=('model', 'split', 'box', 'metric', 'root_ids'))
        .reset_index()
    )

    # Map styles + offset pour le jitter en abscisse
    style_map = {
        'before_expertized': dict(color='orange', linestyle='--', marker=None,
                                  linewidth=2.5, markersize=8, alpha=0.8, xoff=-0.1),
        'expertized': dict(color='green',  linestyle='-',  marker='o',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.0),
        'Prediction': dict(color='red',    linestyle='-',  marker='x',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.1),
    }

    plt.figure(figsize=(16, 8))
    for source, style in style_map.items():
        df_s = df_tmp[df_tmp['source'] == source]
        if df_s.empty:
            continue

        # convertir time en float si nécessaire
        x = pd.to_numeric(df_s['time'], errors='coerce').astype(float)
        # appliquer le jitter
        x = x + style['xoff']

        plt.plot(
            x, df_s['value'],
            label=source,
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=style['linewidth'],
            markersize=style['markersize'],
            alpha=style['alpha']
        )

        # annoter le dernier point
        last_x = x.iloc[-1]
        last_y = df_s['value'].iloc[-1]
        plt.text(
            last_x, last_y,
            f" {source}",
            va='center',
            fontsize=9,
            color=style['color']
        )

    # Remettre les vraies valeurs de time en ticks
    uniq_times = sorted(df_tmp['time'].unique())
    plt.xticks(uniq_times, uniq_times)

    plt.title(f"{metric} – {model} / {split} / {box} / root {root}")
    plt.xlabel('Temps')
    plt.ylabel('Value')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()

In [24]:
model_slider  = widgets.IntSlider(min=0, max=len(models)-1,   step=1, value=0, description='Model')
split_slider  = widgets.IntSlider(min=0, max=len(splits)-1,   step=1, value=0, description='Split')
box_slider    = widgets.IntSlider(min=0, max=4,               step=1, value=0, description='Box')
metric_slider = widgets.IntSlider(min=0, max=len(metrics)-1,  step=1, value=0, description='Metric')

# Slider “root” : on fixe temporairement sa plage max à 0…9 – on mettra à jour dynamiquement
root_slider   = widgets.IntSlider(min=0, max=0,               step=1, value=0, description='Root')

# Callback pour mettre à jour le slider root_ids quand on change l’une des 4 premières valeurs
def update_root_slider(*args):
    roots = get_root_ids(
        models[model_slider.value],
        splits[split_slider.value],
        boxes[box_slider.value + split_slider.value*5],
        metrics[metric_slider.value]
    )
    root_slider.max = len(roots)-1
    if(root_slider.value >= root_slider.max):
        root_slider.value = root_slider.max
    
# On observe les changements
for w in (model_slider, split_slider, box_slider, metric_slider):
    w.observe(update_root_slider, 'value')

# Lancer la première initialisation
update_root_slider()

# 6) Affichage interactif
ui = widgets.HBox([model_slider, split_slider, box_slider, metric_slider, root_slider])
out = widgets.interactive_output(
    plot_root,
    {
        'model_idx':  model_slider,
        'split_idx':  split_slider,
        'box_idx':    box_slider,
        'metric_idx': metric_slider,
        'root_idx':   root_slider
    }
)

display(ui, out)

Output()